In [1]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer
import warnings
warnings.filterwarnings("ignore")
import evaluate
import numpy as np
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer


<frozen importlib._bootstrap>:228: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


In [2]:
train_df = pd.read_csv("C://Users//User//Desktop//titles_data.csv", delimiter=';', skiprows=0, low_memory=False)
train_df=train_df.reset_index(drop=True)
train_df=train_df.rename(columns={"titles": "text", "target": "label"})
train_df

,text,label
0,Родственник раскрыл настоящую фамилию Пугачёво...,1
1,Предсказания Матроны Московской на 2024-й год:...,1
2,"Пророчество схимонахини Нины об антихристе, ми...",1
3,«Думал об этом»: что Путин сказал о своем прее...,1
4,Путин поручил уведомить россиян об изменениях ...,1
...,...,...
3193,Путин поручил передать Республике Крым все акц...,0
3194,ЕК изучит просьбу Венгрии по нарушению Болгари...,0
3195,"Глава ""Россетей"" доложил Путину о достижении ц...",0
3196,"Платформа ""Мой экспорт"" научит устанавливать д...",0


In [3]:
ds = Dataset.from_pandas(train_df)
ds

Dataset({
    features: ['text', 'label'],
    num_rows: 3198
})

In [4]:
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2") 

In [5]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [6]:
tokenized_ds = ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/3198 [00:00<?, ? examples/s]

In [7]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
f1 = evaluate.load("f1")

In [8]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    d = {
        **accuracy.compute(predictions=predictions, references=labels),
        **precision.compute(predictions=predictions, references=labels),
        **f1.compute(predictions=predictions, references=labels)
    }
    return d

In [9]:
id2label = {0: "не кликбейт", 1: "кликбейт"}
label2id = {"не кликбейт": 0, "кликбейт": 1}

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    "cointegrated/rubert-tiny2", num_labels=2, id2label=id2label, label2id=label2id
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
split = tokenized_ds.train_test_split(test_size = 0.2)

In [13]:
training_args = TrainingArguments(
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    remove_unused_columns=True,
    load_best_model_at_end=True,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,F1
1,0.654800,0.355897,0.864062,0.905882,0.841530
2,0.269900,0.210721,0.917188,0.897690,0.911223
3,0.174000,0.188360,0.925000,0.921233,0.918089
4,0.101600,0.190752,0.935937,0.906752,0.932231
5,0.056800,0.187803,0.940625,0.929530,0.935811
6,0.041500,0.204522,0.945312,0.919094,0.941957
7,0.033300,0.238379,0.935937,0.899054,0.932897
8,0.019300,0.240201,0.937500,0.904459,0.934211
9,0.020700,0.240369,0.942187,0.913183,0.938843
10,0.022400,0.237390,0.940625,0.915584,0.936877


TrainOutput(global_step=800, training_loss=0.13010873414576055, metrics={'train_runtime': 45.9323, 'train_samples_per_second': 556.906, 'train_steps_per_second': 17.417, 'total_flos': 8296357319448.0, 'train_loss': 0.13010873414576055, 'epoch': 10.0})

In [12]:
#увеличенные батчи
training_args = TrainingArguments(
    learning_rate=5e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=10,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    remove_unused_columns=True,
    load_best_model_at_end=True,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,F1
1,No log,0.678252,0.682813,0.680000,0.685271
2,No log,0.625366,0.820312,0.811550,0.822804
3,0.658200,0.458927,0.862500,0.871795,0.860759
4,0.658200,0.314649,0.875000,0.863636,0.876923
5,0.343400,0.226963,0.914062,0.912773,0.914197
6,0.343400,0.201811,0.923438,0.927445,0.923077
7,0.343400,0.199143,0.926562,0.930599,0.926217
8,0.127900,0.205207,0.932813,0.926154,0.933333
9,0.127900,0.207806,0.929688,0.915408,0.930876
10,0.073200,0.203866,0.934375,0.926380,0.934985


TrainOutput(global_step=200, training_loss=0.3006850290298462, metrics={'train_runtime': 28.4563, 'train_samples_per_second': 898.923, 'train_steps_per_second': 7.028, 'total_flos': 9181146934584.0, 'train_loss': 0.3006850290298462, 'epoch': 10.0})

In [13]:
trainer.evaluate(split['test'])

{'eval_loss': 0.19914335012435913,
 'eval_accuracy': 0.9265625,
 'eval_precision': 0.9305993690851735,
 'eval_f1': 0.9262166405023547,
 'eval_runtime': 0.4257,
 'eval_samples_per_second': 1503.475,
 'eval_steps_per_second': 11.746,
 'epoch': 10.0}

# Многоклассовая классификация

In [4]:
df = pd.read_csv("C:\\Users\\User\\Desktop\\lenta_ru_news_2019_2023.csv")
df

,url,title,text,topic,tags,date
0,https://lenta.ru/news/2019/12/15/prsm/,Россиянам дали советы по выбору чая,Россиянам дали советы при выборе чая. Рекоменд...,Россия,Общество,2019-12-15
1,https://lenta.ru/news/2019/12/15/fb/,В Госдуме назвали японское заявление о Курилах...,Спикер Госдумы Вячеслав Володин назвал угрозой...,Россия,Политика,2019-12-15
2,https://lenta.ru/news/2019/12/15/kino/,Украинская ЛГБТ-активистка обвинила ню-фотогра...,Украинская ЛГБТ-активистка Виктория Гуйвик обв...,Культура,Фотография,2019-12-15
3,https://lenta.ru/news/2019/12/15/alba/,Полицейские застрелили порезавшего мать буйног...,В Москве полицейские застрелили мужчину при по...,Силовые структуры,Криминал,2019-12-15
4,https://lenta.ru/news/2019/12/15/anons/,Беглого президента Боливии решили арестовать,Исполняющая обязанности президента Боливии Жан...,Мир,Политика,2019-12-15
...,...,...,...,...,...,...
496252,https://lenta.ru/news/2024/01/01/braziliya-pri...,Пушилин сообщил о раненых в результате обстрел...,Глава Донецкой народной республики (ДНР) Денис...,Бывший СССР,Украина,2024-01-01
496253,https://lenta.ru/news/2024/01/01/eks-glava-mid...,В России подняли призывной возраст до 30 лет,С 1 января 2024 года начал действовать закон о...,Россия,Политика,2024-01-01
496254,https://lenta.ru/news/2024/01/01/na-ukraine-vy...,В России изменились условия выдачи материнског...,С 1 января 2024 года материнский капитал будут...,Россия,Политика,2024-01-01
496255,https://lenta.ru/news/2024/01/01/pushilin-soob...,На Украине высказались о возможной смене власти,Директор Украинского института анализа и менед...,Бывший СССР,Украина,2024-01-01


In [5]:
lenta_topics = {
    "Россия": 0,
    "Экономика": 1,
    "Силовые структуры": 2,
    "Бывший СССР": 3,
    "Спорт": 4,
    "Забота о себе": 5,
    "Здоровье": 5,
    "Строительство": 6,
    "Путешествия": 7,
    "Наука и техника": 8,
    "Интернет и СМИ": 0,
    "Бизнес": 1,
    }

In [6]:
df["number"] = df["topic"].apply(lambda x: lenta_topics[x] if x in lenta_topics else None)
df = df.dropna(subset=['number'])
df = df.dropna(subset=['text'])

df["number"] = df["number"].apply(lambda x: int(x))

In [7]:
df.columns = ['url','title','text','topic','tags','date','label']

In [8]:
halfed_df = df.iloc[::2].reset_index(drop=True)

In [9]:
quartered_df = halfed_df.iloc[::2].reset_index(drop=True)

In [10]:
new_ds = Dataset.from_pandas(quartered_df)
new_ds

Dataset({
    features: ['url', 'title', 'text', 'topic', 'tags', 'date', 'label'],
    num_rows: 82335
})

In [11]:
new_ds

Dataset({
    features: ['url', 'title', 'text', 'topic', 'tags', 'date', 'label'],
    num_rows: 82335
})

In [12]:
id2label = {
    0: "Россия", 
    1: "Экономика",
    2: "Силовые структуры", 
    3: "Бывший СССР",
    4: "Спорт", 
    5: "Забота о себе",
    6: "Строительство", 
    7: "Путешествия",
    8: "Наука и техника"
}
label2id = {
    "Россия": 0,
    "Экономика": 1,
    "Силовые структуры": 2,
    "Бывший СССР": 3,
    "Спорт": 4,
    "Забота о себе": 5,
    "Строительство": 6,
    "Путешествия": 7,
    "Наука и техника": 8,
    }

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    "cointegrated/rubert-tiny2", num_labels=9, id2label=id2label, label2id=label2id
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
tokenized_new_ds = new_ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/82335 [00:00<?, ? examples/s]

In [15]:
new_split = tokenized_new_ds.train_test_split(test_size = 0.2)
new_split

DatasetDict({
    train: Dataset({
        features: ['url', 'title', 'text', 'topic', 'tags', 'date', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 65868
    })
    test: Dataset({
        features: ['url', 'title', 'text', 'topic', 'tags', 'date', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16467
    })
})

In [16]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

In [17]:
def compute_metrics_multi(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    d = {
        **accuracy.compute(predictions=predictions, references=labels),
        **precision.compute(predictions=predictions, references=labels, average="weighted"),
        **recall.compute(predictions=predictions, references=labels, average="weighted"),
        **f1.compute(predictions=predictions, references=labels, average="weighted")
    }
    return d

In [18]:
training_args = TrainingArguments(
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    remove_unused_columns=True,
    load_best_model_at_end=True,
    logging_steps=50,
    
    gradient_accumulation_steps=2,
    optim="adafactor",
    torch_compile=False,
    fp16=True,
    fp16_full_eval=True,
    gradient_checkpointing=True,
    auto_find_batch_size=True,
    
    dataloader_num_workers=4,
    dataloader_pin_memory=True,

)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=new_split['train'],
    eval_dataset=new_split['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_multi,
)

trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.247800,0.269455,0.912067,0.913478,0.912067,0.912052
2,0.155000,0.274279,0.919475,0.920184,0.919475,0.919497
3,0.099800,0.389762,0.917836,0.918773,0.917836,0.917804
4,0.088300,0.469236,0.917957,0.917952,0.917957,0.917930
5,0.011400,0.500501,0.916925,0.916945,0.916925,0.916903


TrainOutput(global_step=20585, training_loss=0.1387825949800255, metrics={'train_runtime': 37530.0163, 'train_samples_per_second': 8.775, 'train_steps_per_second': 0.548, 'total_flos': 1945860139971648.0, 'train_loss': 0.1387825949800255, 'epoch': 5.0})

In [20]:
trainer.evaluate(new_split['test'])

{'eval_loss': 0.2694593071937561,
 'eval_accuracy': 0.9120665573571385,
 'eval_precision': 0.9134778738208919,
 'eval_recall': 0.9120665573571385,
 'eval_f1': 0.9120517011301996,
 'eval_runtime': 727.8481,
 'eval_samples_per_second': 22.624,
 'eval_steps_per_second': 1.415,
 'epoch': 5.0}